In [ ]:
import os
import gspread
import pandas as pd
import time
import sys

from dotenv import load_dotenv

load_dotenv()

In [ ]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "")
os.environ["NO_PROXY"] = os.getenv("NO_PROXY", "")

print("Proxy configurado.")

In [ ]:
credencial_google = os.getenv("GOOGLE_CREDENTIALS_PATH")

if not credencial_google:raise ValueError("GOOGLE_CREDENTIALS_PATH não foi encontrado no .env.")
if not os.path.isfile(credencial_google):raise FileNotFoundError(f"Credencial Google não encontrada: {credencial_google}")

gp = gspread.service_account(filename=credencial_google)

print("Credencial Google encontrada.")
print("Conexão com Google Sheets pronta.")

In [ ]:
print("Abrindo Planilha")

planilha = gp.open('teste_alunos')
aba = planilha.sheet1 

dados_puros = aba.get_all_values()

df = pd.DataFrame(dados_puros[1:], columns=dados_puros[0])

In [ ]:
display(df)

In [ ]:
if "Status" not in df.columns:
    raise ValueError("A coluna 'Status' não foi encontrada na planilha.")

pendentes = df[df["Status"] == "Pendente"]

display(pendentes)

quantidade_pendentes = len(pendentes)

print(f"Quantidade de alunos pendentes: {quantidade_pendentes}")

In [ ]:
nome_aba = "Pendentes"

try:
    aba_pendentes = planilha.worksheet(nome_aba)
    print(f"A aba '{nome_aba}' já existe.")
except gspread.WorksheetNotFound:
    aba_pendentes = planilha.add_worksheet(title=nome_aba,rows=100,cols=10)
    print(f"A aba '{nome_aba}' foi criada.")

In [ ]:
dados_pendentes = [pendentes.columns.tolist()] + pendentes.astype(str).values.tolist()
dados_pendentes

In [ ]:
aba_pendentes.clear()
aba_pendentes.update(range_name="A1",values=dados_pendentes)
print("Dados pendentes gravados com sucesso.")